# **Policy Assistant Using RAG & HuggingFace**

Built an AI-powered question-answering system designed to help employees instantly find answers from company policy documents. Instead of manually searching through pages of text, employees can simply type a question and get a clear, accurate answer in seconds.

To make this work, combined two powerful techniques — retrieval and generation. First, the system searches through the documents to find the most relevant paragraphs using Facebook's Dense Passage Retriever (DPR) and FAISS for fast similarity search. Then, it feeds those paragraphs into GPT-2, which generates a natural, readable answer based on the retrieved content.

Built the full pipeline from scratch — loading and preprocessing the documents, encoding them into vector embeddings, indexing them for fast search, and finally connecting everything to the GPT-2 generator. The result is a system that gives grounded, document-backed answers rather than relying on what the AI already knows, making it far more accurate and trustworthy for domain-specific use cases.

### **Import Libraries**


In [19]:
import random
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer, DPRQuestionEncoder, DPRQuestionEncoderTokenizer
import wget

### Load Dataset 

In [2]:
file_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

In [4]:
filename = "d:/Projects/rag-policy-assistant/files/company_policies.txt"
wget.download(file_url, out = filename)


'd:/Projects/rag-policy-assistant/files/company_policies.txt'

##### Pre-process Data

In [15]:
def data_preprocess (filename):
    
    with open (filename, "r") as file:
        text = file.read()
    
    paragraphs = text.split('\n')
    
    paragraph_list = [para.strip() for para in paragraphs if len(para.strip())> 0]
    
    return paragraph_list

In [17]:
policy_list = data_preprocess(filename=filename)
policy_list[0:3]

['1.\tCode of Conduct',
 'Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.',
 'Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.']

## Building the Retriever: Encoding and Indexing

Computers can't understand raw text directly — they work with numbers. So the first step is converting each paragraph into a list of numbers called a **vector embedding**. These vectors capture the meaning of the text, making it possible to compare paragraphs and find the ones most relevant to a user's question.

In this section, we encode all the paragraphs into vector embeddings and store them in a **FAISS index** — a tool that lets us search through thousands of vectors very quickly to find the best matches for any given query.

## Encoding Texts into Embeddings

To encode the paragraphs, we use the **Dense Passage Retriever (DPR)** model, specifically its context encoder. DPR is built on top of BERT but trained differently — instead of handling general NLP tasks, it is trained specifically to retrieve relevant passages by learning which documents best match a given question. This makes it much better at capturing the kind of meaning that matters for search and retrieval, compared to using a general-purpose BERT model.

In [18]:
model_name = "facebook/dpr-ctx_encoder-single-nq-base"

In [20]:
# context tokenizer
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\models--facebook--dpr-ctx_encoder-single-nq-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [21]:
context_tokenizer

DPRContextEncoderTokenizer(name_or_path='facebook/dpr-ctx_encoder-single-nq-base', vocab_size=30522, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})